# 当前 ConvNeXt 二分类器可视化

这个 notebook 用当前 `binary_classification` 代码和部署中的 ConvNeXt-Small 权重完成三件事：

1. 核对 H5 样本与验证阶段的预处理；
2. 并排查看正负样本、预测概率和误分类；
3. 用 Grad-CAM 查看模型把哪些 time–DM 区域当作正类证据。

它替代旧 `VisualData*.ipynb` 中已经失效的 ResNet/CenterNet 路径，不负责重新训练或计算完整验证集指标。


In [ ]:
# fmt: off

from __future__ import annotations

import hashlib
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display


def find_project_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = (
        cwd,
        cwd / "binary_classification",
        cwd / "bssearch" / "binary_classification",
    )
    for candidate in candidates:
        if (candidate / "binary_model.py").is_file() and (
            candidate / "binary_data.py"
        ).is_file():
            return candidate
    raise FileNotFoundError(
        "请从 LPT、bssearch 或 binary_classification 目录启动 notebook。"
    )


def file_sha256(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


SEED        = 42
N_PER_CLASS = 3
MODEL_TYPE  = "ConvNeXtNet"
MODEL_NAME  = "convnext_small"
PROJECT_DIR = find_project_dir()
DATA_DIR    = PROJECT_DIR / "Data"
WEIGHTS_PATH = (
    PROJECT_DIR.parent
    / "search_pipeline"
    / "models"
    / "binary_best_model_conv_small_ema.pth"
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert DATA_DIR.is_dir(), DATA_DIR
assert WEIGHTS_PATH.is_file(), WEIGHTS_PATH
sys.path.insert(0, str(PROJECT_DIR))

from binary_data import BurstDataset, build_catalog  # noqa: E402
from binary_infer import load_weights                # noqa: E402
from binary_model import build_binary_model          # noqa: E402

np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="white")

{
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "weights": str(WEIGHTS_PATH),
    "weights_sha256": file_sha256(WEIGHTS_PATH),
    "device": str(DEVICE),
}

# fmt: on

## 实验计划

- 固定随机种子，从去重后的 catalog 分别抽取正、负样本。
- 首先逐步展示当前验证预处理，确认图像方向和动态范围。
- 然后加载生产搜索使用的 EMA checkpoint，展示正类概率。
- 最后对正类 logit 做 Grad-CAM；热区应落在模型实际用于判断的结构上，而不是边缘或整幅背景。


In [ ]:
# fmt: off

catalog = build_catalog(str(DATA_DIR))
catalog_summary = (
    catalog.assign(class_name=catalog["label"].map({0: "negative", 1: "positive"}))
    .groupby(["class_name", "source"], dropna=False)
    .size()
    .rename("samples")
    .reset_index()
)
display(catalog_summary)

selected = pd.concat(
    [
        catalog[catalog["label"] == label].sample(
            n            = min(N_PER_CLASS, int((catalog["label"] == label).sum())),
            random_state = SEED + label,
        )
        for label in (1, 0)
    ],
    ignore_index=True,
)
selected["class_name"] = selected["label"].map({0: "negative", 1: "positive"})
display(selected[["class_name", "name", "source", "h5_file", "local_idx"]])

dataset = BurstDataset(selected, val=True)

# fmt: on

## 当前验证预处理

下面复现 `BurstDataset.preprocess_data` 的固定验证路径：原图加 1、按频率列均值去基线、5%–95% 截断，最后归一化到 `[0, 1]`。末尾断言确保这里展示的结果与模型实际输入一致。


In [ ]:
# fmt: off

def load_raw(row: pd.Series) -> np.ndarray:
    with h5py.File(row["h5_file"], "r") as handle:
        return handle["images"][int(row["local_idx"])].astype(np.float32)


row              = selected.iloc[0]
raw              = load_raw(row)
shifted          = raw + 1.0
baseline_removed = shifted / np.mean(shifted, axis=0)
vmin, vmax       = np.nanpercentile(baseline_removed, [5, 95])
clipped          = np.clip(baseline_removed, vmin, vmax)
normalized       = (clipped - clipped.min()) / (clipped.max() - clipped.min() + 1e-8)
model_input, _   = dataset[0]
np.testing.assert_allclose(normalized, model_input.squeeze(0).numpy(), atol=1e-6)

panels = [
    ("Raw H5", raw),
    ("Baseline removed", baseline_removed),
    ("5–95% clipped", clipped),
    ("Model input", normalized),
]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), constrained_layout=True)
cmap      = sns.color_palette("mako", as_cmap=True)
for axis, (title, image) in zip(axes, panels):
    lo, hi = np.nanpercentile(image, [1, 99])
    axis.imshow(image, origin="lower", aspect="auto", cmap=cmap, vmin=lo, vmax=hi)
    axis.set_title(title)
    axis.set_xlabel("DM index")
    axis.set_yticks([])
axes[0].set_ylabel("Time index")
fig.suptitle(f"{row['class_name']}: {row['name']}")
plt.show()

# fmt: on

## 新模型预测

这里直接调用当前训练/推理模块的模型构造和权重加载函数，避免 notebook 内再维护一份模型定义。图标题中的 `p(burst)` 是 softmax 正类概率。


In [ ]:
# fmt: off

model = build_binary_model(
    model_type  = MODEL_TYPE,
    model_name  = MODEL_NAME,
    num_classes = 2,
    pretrained  = False,
).to(DEVICE)
load_weights(model, str(WEIGHTS_PATH), DEVICE)
model.eval()

images    = []
prob_true = []
predicted = []
with torch.no_grad():
    for index in range(len(dataset)):
        image, _    = dataset[index]
        probability = torch.softmax(model(image.unsqueeze(0).to(DEVICE)), dim=1)[0]
        images.append(image)
        prob_true.append(float(probability[1].cpu()))
        predicted.append(int(probability.argmax().cpu()))

predictions              = selected[["class_name", "name", "source", "label"]].copy()
predictions["predicted"] = predicted
predictions["prob_true"] = prob_true
predictions["correct"]   = predictions["label"] == predictions["predicted"]
display(predictions)

# fmt: on

In [ ]:
# fmt: off

cols      = 3
rows      = int(np.ceil(len(images) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 3.6 * rows), constrained_layout=True)
axes      = np.atleast_1d(axes).ravel()
for axis, image, (_, result) in zip(axes, images, predictions.iterrows()):
    axis.imshow(image.squeeze(0).numpy(), origin="lower", aspect="auto", cmap=cmap)
    status = "correct" if result["correct"] else "wrong"
    axis.set_title(
        f"true={result['class_name']} | p(burst)={result['prob_true']:.3f} | {status}"
    )
    axis.set_xlabel(str(result["name"])[:55])
    axis.set_xticks([])
    axis.set_yticks([])
for axis in axes[len(images) :]:
    axis.axis("off")
plt.show()

# fmt: on

## Grad-CAM：模型关注区域

Grad-CAM 取 ConvNeXt 最后一层二维特征，对正类 logit 反向传播。红色区域表示对 `burst` 判断贡献较大的位置；它是解释性诊断，不是目标框或概率校准结果。


In [ ]:
# fmt: off

def grad_cam_for_positive_class(model: torch.nn.Module, image: torch.Tensor):
    captured = {}

    def capture_activation(_module, _inputs, output):
        captured["activation"] = output
        output.register_hook(
            lambda gradient: captured.__setitem__("gradient", gradient)
        )

    target_layer = model.base_model.features[-1]
    hook         = target_layer.register_forward_hook(capture_activation)
    try:
        model.zero_grad(set_to_none=True)
        logits      = model(image.unsqueeze(0).to(DEVICE))
        probability = torch.softmax(logits, dim=1)[0, 1]
        logits[0, 1].backward()
    finally:
        hook.remove()

    activation = captured["activation"].detach()
    gradient   = captured["gradient"].detach()
    weights    = gradient.mean(dim=(2, 3), keepdim=True)
    cam        = torch.relu((weights * activation).sum(dim=1, keepdim=True))
    cam = F.interpolate(
        cam, size=image.shape[-2:], mode="bilinear", align_corners=False
    )[0, 0]
    cam -= cam.min()
    cam /= cam.max().clamp_min(1e-8)
    return cam.cpu().numpy(), float(probability.detach().cpu())


example_ids = [
    int(predictions.index[predictions["label"] == 1][0]),
    int(predictions.index[predictions["label"] == 0][0]),
]
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for row_id, sample_id in enumerate(example_ids):
    image                = images[sample_id]
    heatmap, probability = grad_cam_for_positive_class(model, image)
    base                 = image.squeeze(0).numpy()
    axes[row_id, 0].imshow(base, origin="lower", aspect="auto", cmap=cmap)
    axes[row_id, 0].set_title("Model input")
    axes[row_id, 1].imshow(heatmap, origin="lower", aspect="auto", cmap="inferno")
    axes[row_id, 1].set_title("Positive-class Grad-CAM")
    axes[row_id, 2].imshow(base, origin="lower", aspect="auto", cmap="gray")
    axes[row_id, 2].imshow(
        heatmap, origin="lower", aspect="auto", cmap="inferno", alpha=0.45
    )
    axes[row_id, 2].set_title(f"overlay | p(burst)={probability:.3f}")
    for axis in axes[row_id]:
        axis.set_xticks([])
        axis.set_yticks([])
    axes[row_id, 0].set_ylabel(predictions.loc[sample_id, "class_name"])
plt.show()

# fmt: on

## 结果记录与下一步

运行后重点检查：

- 预处理四联图的方向、裁剪和归一化是否符合预期；
- 高置信误分类是否集中在某一来源或某类干扰；
- Grad-CAM 是否落在 burst 轨迹/局部结构，而不是边缘、空白或整幅背景；
- 如需完整指标，使用 `binary_infer.py` 导出全量预测表，不在本 notebook 中重复实现评测。
